## 1. Setup & Imports

In [ ]:
import os
import glob
import json
import math
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn.utils.rnn as rnn_utils
from torch.cuda.amp import autocast, GradScaler
import pandas as pd
import sentencepiece as spm
from datasets import load_dataset

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Dataset

In [ ]:
ds = load_dataset("cfilt/iitb-english-hindi")
print(ds)

def build_df(split):
    df = ds[split].to_pandas()
    df['english'] = df['translation'].apply(lambda x: x['en'])
    df['hindi']   = df['translation'].apply(lambda x: x['hi'])
    return df[['english', 'hindi']]

train_df = build_df('train')
val_df   = build_df('validation')
test_df  = build_df('test')

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
train_df.head(3)

## 3. Load BPE Tokenizer from Kaggle Input

In [ ]:

matches = glob.glob("/kaggle/input/**/bpe.model", recursive=True)

if matches:
    BPE_MODEL_PATH = matches[0]
    print(f"Found bpe.model at: {BPE_MODEL_PATH}")
else:
    raise FileNotFoundError(
        "bpe.model not found"
    )

sp = spm.SentencePieceProcessor()
sp.load(BPE_MODEL_PATH)

print(f"Vocab size : {sp.vocab_size()}")
print(f"BOS id     : {sp.bos_id()}")
print(f"EOS id     : {sp.eos_id()}")
print(f"PAD id     : {sp.pad_id()}")
print(f"UNK id     : {sp.unk_id()}")

test_enc = sp.encode("Hello world", out_type=int)
test_dec = sp.decode(test_enc)
print(f"\nSanity check: 'Hello world' → {test_enc} → '{test_dec}'")

## 4. Encode & Filter Data

In [ ]:
PAD_ID = 0
BOS_ID = 2
EOS_ID = 3

# Filter out very long sequences 
MAX_LEN = 100

def encode_text(text):
    return [BOS_ID] + sp.encode(str(text), out_type=int) + [EOS_ID]

def encode_df(df):
    df = df.copy()
    df['en_ids'] = df['english'].apply(encode_text)
    df['hi_ids'] = df['hindi'].apply(encode_text)
    # Drop pairs where either side exceeds MAX_LEN tokens
    mask = df['en_ids'].apply(len).le(MAX_LEN) & df['hi_ids'].apply(len).le(MAX_LEN)
    return df[mask].reset_index(drop=True)

print("Encoding training data...")
train_df = encode_df(train_df)
print("Encoding validation data...")
val_df   = encode_df(val_df)
print("Encoding test data...")
test_df  = encode_df(test_df)

print(f"After length filter → Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
train_df.head(2)

## 5. Dataset Class & Collate Function

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.source = df['en_ids'].tolist()
        self.target = df['hi_ids'].tolist()

    def __len__(self):
        return len(self.source)

    def __getitem__(self, index):
        return (
            torch.tensor(self.source[index], dtype=torch.long),
            torch.tensor(self.target[index], dtype=torch.long)
        )


def collate_fn(batch):
    source_batch, target_batch = zip(*batch)
    source_batch = rnn_utils.pad_sequence(source_batch, batch_first=True, padding_value=PAD_ID)
    target_batch = rnn_utils.pad_sequence(target_batch, batch_first=True, padding_value=PAD_ID)
    return source_batch, target_batch

## 6. DataLoaders

In [ ]:

BATCH_SIZE       = 16
GRAD_ACCUM_STEPS = 4

df_small = train_df.sample(n=200_000, random_state=42).reset_index(drop=True)

small_train_dataset = CustomDataset(df_small)
val_dataset         = CustomDataset(val_df)
test_dataset        = CustomDataset(test_df)

small_train_dataLoader = DataLoader(
    small_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)
val_dataLoader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)
test_dataLoader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Train batches : {len(small_train_dataLoader):,}")
print(f"Val batches   : {len(val_dataLoader)}")
print(f"Test batches  : {len(test_dataLoader)}")

## 7. Transformer Model

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding with dropout """
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
      
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class Custom_Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=6, dim_ff=1024, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        self.embedding         = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.positional_encoding = PositionalEncoding(d_model, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        self.fully_connected = nn.Linear(d_model, vocab_size)

     
        self.fully_connected.weight = self.embedding.weight

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, source, target):
        # Padding masks
        src_key_padding_mask = (source == PAD_ID)
        tgt_key_padding_mask = (target == PAD_ID)

        # repeating tokens fix: causal mask stops decoder from seeing future tokens
        tgt_seq_len = target.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt_seq_len, device=source.device
        )

        # loss stuck fix 
        source = self.positional_encoding(self.embedding(source) * math.sqrt(self.d_model))
        target = self.positional_encoding(self.embedding(target) * math.sqrt(self.d_model))

        output = self.transformer(
            source,
            target,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.fully_connected(output)

## 8. Model, Loss, Optimizer & Scheduler

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = Custom_Transformer(
    vocab_size=28000,
    d_model=256,
    nhead=4,
    num_layers=6,
    dim_ff=1024,
    dropout=0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


loss_fun = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1.0,           # LR fully controlled by Noam scheduler
    betas=(0.9, 0.98),
    eps=1e-9
)
D_MODEL      = 256
WARMUP_STEPS = 4000

def noam_lambda(step):
    step = max(step, 1)
    return (D_MODEL ** -0.5) * min(step ** -0.5, step * WARMUP_STEPS ** -1.5)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=noam_lambda)

# mixed precision scaler
scaler = GradScaler()

print("Model, loss, optimizer, scheduler and AMP scaler ready.")

## 9. Evaluate Function

In [ ]:
def evaluate(model, val_dataLoader, loss_fun, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for source, target in val_dataLoader:
            source = source.to(device)
            target = target.to(device)
            with autocast():
                output = model(source, target[:, :-1])
                output = output.reshape(-1, output.shape[-1])
                tgt_y  = target[:, 1:].reshape(-1)
                loss   = loss_fun(output, tgt_y)
            total_loss += loss.item()
    return total_loss / len(val_dataLoader)

## 10. Training Loop

In [ ]:
import math as _math

NUM_EPOCHS    = 10
LOG_EVERY     = 200
best_val_loss = float("inf")

# Kaggle output directory
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.cuda.empty_cache()

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

   
    torch.save(model.state_dict(), f"{OUTPUT_DIR}/checkpoint_epoch{epoch}.pt")

    for i, (source, target) in enumerate(small_train_dataLoader, 1):

        source = source.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        #  autocast wraps the forward pass for FP16 computation
        with autocast():
            output       = model(source, target[:, :-1])
            output       = output.reshape(-1, output.shape[-1])
            target_shift = target[:, 1:].reshape(-1)
            loss         = loss_fun(output, target_shift)
            loss         = loss / GRAD_ACCUM_STEPS  # scale for accumulation

        
        scaler.scale(loss).backward()

        # Gradient accumulation, only step optimizer every N mini-batches
        if i % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)

            # detect exploding gradients before they corrupt weights
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if torch.isnan(total_norm) or torch.isinf(total_norm):
                print(f"  NaN/Inf gradient at epoch {epoch+1} batch {i} — skipping step")
                optimizer.zero_grad()
                scaler.update()
                continue

            scaler.step(optimizer)
            scaler.update()
            # step warmup scheduler every optimizer step
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM_STEPS  

        if i % LOG_EVERY == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(
                f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                f"Batch {i}/{len(small_train_dataLoader)} | "
                f"Loss {loss.item() * GRAD_ACCUM_STEPS:.4f} | "
                f"LR {current_lr:.6f}"
            )

    train_loss = total_loss / len(small_train_dataLoader)

    # if train_loss is NaN, restore from this epoch's checkpoint and skip
    if _math.isnan(train_loss):
        print(f"   NaN detected in epoch {epoch+1}! Restoring from checkpoint...")
        model.load_state_dict(torch.load(f"{OUTPUT_DIR}/checkpoint_epoch{epoch}.pt"))
        print(f"   Restored to start-of-epoch-{epoch+1} weights. Continuing...")
        continue

    val_loss = evaluate(model, val_dataLoader, loss_fun, device)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1} complete | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"{'='*60}\n")

    # Save best model to Kaggle output
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_transformer_model.pt")
        print(f"  Best model saved (val_loss={val_loss:.4f})\n")

print("Training complete!")
print(f"Best val loss: {best_val_loss:.4f}")

## 11. Save All Outputs to /kaggle/working

In [ ]:
# Save BPE model alongside the trained model for easy reuse
shutil.copy(BPE_MODEL_PATH, f"{OUTPUT_DIR}/bpe.model")

# Save training metadata
training_info = {
    "best_val_loss"     : best_val_loss,
    "epochs_trained"    : NUM_EPOCHS,
    "vocab_size"        : sp.vocab_size(),
    "d_model"           : D_MODEL,
    "warmup_steps"      : WARMUP_STEPS,
    "batch_size"        : BATCH_SIZE,
    "grad_accum_steps"  : GRAD_ACCUM_STEPS,
    "max_seq_len"       : MAX_LEN,
}
with open(f"{OUTPUT_DIR}/training_info.json", "w") as f:
    json.dump(training_info, f, indent=2)

print("Saved to /kaggle/working/:")
for fname in os.listdir(OUTPUT_DIR):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = os.path.getsize(fpath) / 1e6
    print(f"   {fname:45s} {size:.1f} MB")

## 12. Inference

In [ ]:
# Load best saved model
model.load_state_dict(
    torch.load(f"{OUTPUT_DIR}/best_transformer_model.pt", map_location=device)
)
model.eval()
print("Best model loaded.")


def translate_sentence(sentence, model, sp, device, max_len=80):
    """Greedy-decode an English sentence into Hindi."""
    model.eval()
    tokens = [BOS_ID] + sp.encode(sentence, out_type=int) + [EOS_ID]
    src    = torch.tensor([tokens], dtype=torch.long).to(device)
    tgt_tokens = [sp.bos_id()]

    with torch.no_grad():
        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_tokens], dtype=torch.long).to(device)
            output     = model(src, tgt_tensor)
            next_token = output[0, -1].argmax().item()
            tgt_tokens.append(next_token)
            if next_token == sp.eos_id():
                break

    return sp.decode(tgt_tokens[1:-1])


# Smoke tests
test_sentences = [
    "How are you?",
    "The weather is very nice today.",
    "I want to learn machine learning.",
    "Please open the door.",
    "India is a great country."
]

print("\nTranslation results:")
print("-" * 50)
for s in test_sentences:
    hi = translate_sentence(s, model, sp, device)
    print(f"EN: {s}")
    print(f"HI: {hi}")
    print()

In [ ]:
import matplotlib.pyplot as plt

steps = range(1, 20001)
lrs   = [noam_lambda(s) for s in steps]

plt.figure(figsize=(9, 3))
plt.plot(steps, lrs)
plt.axvline(WARMUP_STEPS, color='red', linestyle='--', label=f'warmup={WARMUP_STEPS}')
plt.title("Noam LR Schedule (d_model=256, warmup=4000)")
plt.xlabel("Optimizer step")
plt.ylabel("LR multiplier")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/lr_schedule.png", dpi=120)
plt.show()
print("Peak LR step:", lrs.index(max(lrs)) + 1, " | Peak value:", round(max(lrs), 6))